## LangGraph: execução paralela de nodes

Agora vamos trabalhar com 3 nodes paralelamente enviando informações para um mesmo estado


In [3]:
def unicos(atual : list, novo :list)-> list:
    return list(dict.fromkeys((atual or [])+(novo or [])))
    
def mesclar(atual : dict, novo :dict)-> dict:
    return {
        **(atual or {}),
        **(novo or {})}

def maior(atual:float, novo:float)->float:
    return max((atual or 0),(novo or 0))

def ultimos(n: int):
    def reducer(atual:list,novo:list)->list:
        return ((atual or [])+(novo or []))[-n:]
    
    return reducer

### Estado (typedDict)
Cada avaliador dá uma nota para um fornecedor. Queremos:
- `fontes`: lista de fontes **sem duplicadas**
- `notas`: um **dict** `{avaliador: nota}` com a contribuição de todos
- `melhor_nota`: a **maior** nota
- `historico`: só os **últimos 3** eventos

In [4]:
from typing import TypedDict,Annotated

In [11]:
class Avaliacao(TypedDict):
    fornecedor : str
    fontes : Annotated[list[str],unicos]
    notas : Annotated[dict[str:float],mesclar]
    maior_nota : Annotated[float,maior]
    historico : Annotated[list[str],ultimos(3)]

### Fabrica de Nodes

In [29]:
def avaliador(nome:str,nota:float,fontes:list):
    def node(state: Avaliacao):
        return {
            'fontes' : fontes,
            'notas': {nome:nota},
            'maior_nota' : nota,
            'historico' : [f'{nome} avaliou']
        }
    return node

### Criação do Grafo

In [30]:
from langgraph.graph import StateGraph,START,END

In [31]:
grafo = StateGraph(Avaliacao)
# nodes
grafo.add_node('preco',avaliador('avaliador_preco',7.9,['sites','planilhas']))
grafo.add_node('qualidade',avaliador('avaliador_qualidade',9.0,['sites','auditoria']))
grafo.add_node('prazo', avaliador('avaliador_prazo', 6.0, ['planilha', 'erp']))
grafo.add_node('consolidar',lambda s: {'historico' : ['conslidado']})

#Arestas
for node in ('preco','qualidade','prazo'):
    grafo.add_edge(START,node)
    grafo.add_edge(node,'consolidar')
grafo.add_edge('consolidar',END)


### Vamos executar o grafo

In [32]:
resultado = grafo.compile().invoke({"fornecedor": "ACME", "fontes": [], "notas": {}, "maior_nota": 0, "historico": ["início"]})
print(resultado)

{'fornecedor': 'ACME', 'fontes': ['planilha', 'erp', 'sites', 'planilhas', 'auditoria'], 'notas': {'avaliador_prazo': 6.0, 'avaliador_preco': 7.9, 'avaliador_qualidade': 9.0}, 'maior_nota': 9.0, 'historico': ['avaliador_preco avaliou', 'avaliador_qualidade avaliou', 'conslidado']}
